In [27]:
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

import pandas as pd

In [70]:
def bitinfo_to_smarts(bitinfo, mol):
    bit2smarts = {}
    for bit, info in bitinfo.items():
        bit2smarts[bit] = set()
        for atom_id, radius in info:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_id)
            atomsToUse = set((atom_id, ))
            for b in env:
                atomsToUse.add(mol.GetBondWithIdx(b).GetBeginAtomIdx())
                atomsToUse.add(mol.GetBondWithIdx(b).GetEndAtomIdx())

            enlargedEnv = set()
            for atom in atomsToUse:
                a = mol.GetAtomWithIdx(atom)
                for b in a.GetBonds():
                    bidx = b.GetIdx()
                    if bidx not in env:
                        enlargedEnv.add(bidx)

            enlargedEnv = list(enlargedEnv)
            enlargedEnv += env

            # find all relevant neighbors
            anyAtoms = []
            for a in atomsToUse:
                neighbors = mol.GetAtomWithIdx(a).GetNeighbors()
                for n in neighbors:
                    anyIdx = n.GetIdx()
                    if anyIdx not in atomsToUse:
                        anyAtoms.append(anyIdx)

            # replace atomic number to zero (there is no number for any atom)
            # have to make a sacrificial copy of the molecule here
            alt_mol = Chem.Mol(mol)
            for aA in anyAtoms:
                alt_mol.GetAtomWithIdx(aA).SetAtomicNum(0)

            submol = Chem.PathToSubmol(alt_mol, enlargedEnv)
            del alt_mol

            # change [0] to *
            morgan_bit_smarts = Chem.MolToSmarts(submol).replace('[#0]', '*')
            bit2smarts[bit].add(morgan_bit_smarts)
    return bit2smarts



In [71]:
df = pd.read_csv('AqSolDB_v1.0_min.csv', index_col="ID")
df['mol'] = df['SMILES'].apply(lambda x: Chem.MolFromSmiles(x))
df = df[df['mol'].notna()]
df

,Name,InChIKey,SMILES,Solubility,mol
ID,,,,,
A-3,"N,N,N-trimethyloctadecan-1-aminium bromide",SZEMGTQCPRNXEG-UHFFFAOYSA-M,[Br-].CCCCCCCCCCCCCCCCCC[N+](C)(C)C,-3.616127,<rdkit.Chem.rdchem.Mol object at 0x7277a216fbc0>
A-4,Benzo[cd]indol-2(1H)-one,GPYLCFQEKPUWLD-UHFFFAOYSA-N,O=C1Nc2cccc3cccc1c23,-3.254767,<rdkit.Chem.rdchem.Mol object at 0x7277a216fc30>
A-5,4-chlorobenzaldehyde,AVPYQKSLYISFPO-UHFFFAOYSA-N,Clc1ccc(C=O)cc1,-2.177078,<rdkit.Chem.rdchem.Mol object at 0x7277a216fca0>
A-8,"zinc bis[2-hydroxy-3,5-bis(1-phenylethyl)benzo...",XTUPUYCJWKHGSW-UHFFFAOYSA-L,[Zn++].CC(c1ccccc1)c2cc(C(C)c3ccccc3)c(O)c(c2)...,-3.924409,<rdkit.Chem.rdchem.Mol object at 0x7277a216fd10>
A-9,4-({4-[bis(oxiran-2-ylmethyl)amino]phenyl}meth...,FAUAZXVRLVIARB-UHFFFAOYSA-N,C1OC1CN(CC2CO2)c3ccc(Cc4ccc(cc4)N(CC5CO5)CC6CO...,-4.662065,<rdkit.Chem.rdchem.Mol object at 0x7277a216fd80>
...,...,...,...,...,...
I-84,tetracaine,GKCBAIGFKIBETG-UHFFFAOYSA-N,C(c1ccc(cc1)NCCCC)(=O)OCCN(C)C,-3.010000,<rdkit.Chem.rdchem.Mol object at 0x7277a20bae30>
I-85,tetracycline,OFVLGDICTFRJMM-WESIUVDSSA-N,OC1=C(C(C2=C(O)[C@@](C(C(C(N)=O)=C(O)[C@H]3N(C...,-2.930000,<rdkit.Chem.rdchem.Mol object at 0x7277a20baea0>
I-86,thymol,MGSRCZKZVOBKFT-UHFFFAOYSA-N,c1(cc(ccc1C(C)C)C)O,-2.190000,<rdkit.Chem.rdchem.Mol object at 0x7277a20baf10>


In [ ]:
bit2substructure = [set() for _ in range(2048)]
substructure2count = {}

fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2)
fpgen_meta = rdFingerprintGenerator.AdditionalOutput()
fpgen_meta.AllocateBitInfoMap()

for i, row in df[:10].iterrows():
    mol = row['mol']
    fp = fpgen.GetFingerprint(mol, additionalOutput=fpgen_meta)
    meta = fpgen_meta.GetBitInfoMap()
    for bit, smarts in bitinfo_to_smarts(meta, mol):
        bit2substructure[bit].add(smarts)
        substructure2count[smarts] = substructure2count.get(smarts, 0) + 1

print(bit2substructure)

{80: {'*-[#6]-[#0+]', '*-[#6]-*'}, 294: {'[#6]-[#6]-*'}, 319: {'*-[#6]-[#7+](-[#6])(-[#6])-[#6]'}, 414: {'*-[#7+](-*)(-*)-*'}, 550: {'*-[#6]-[#6]-[#6]-[#7+](-[#6])(-[#6])-[#6]'}, 591: {'[#6]-[#6]-[#6]-[#6]-*'}, 628: {'*-[#7+](-*)(-[#6])-*', '*-[#7+](-*)(-*)-[#6]', '*-[#7+](-[#6])(-*)-*'}, 780: {''}, 794: {'[#6]-[#6]-[#6]-*'}, 913: {'*-[#6]-[#6]-[#7+](-*)(-*)-*'}, 1057: {'[#6]-*', '[#0+]-[#6]'}, 1143: {'*-[#6]-[#6]-[#6]-[#6]-[#6]-[#0+]', '*-[#6]-[#6]-[#6]-[#6]-[#6]-*'}, 1312: {'*-[#6]-[#6]-[#6]-[#6]-[#7+](-*)(-*)-*'}, 1360: {'*-[#6]-[#6]-[#7+](-[#6])(-[#6])-[#6]'}, 1444: {'[#6]-[#6]-[#6]-[#6]-[#6]-*'}, 1911: {'*-[#6]-[#6]-[#6]-[#0+]', '*-[#6]-[#6]-[#6]-*'}}
{314: {'[#8]=[#6](-*)-*'}, 352: {'*=[#6](-*)-[#6](:[#6]:*):[#6](:*):*'}, 527: {'[#8]=[#6]1-[#7]-[#6](:[#6]:*):[#6](:*):[#6]-1:*'}, 650: {'[#8]=*'}, 725: {'*-[#6]1:*:[#6]:[#6]:[#6]2:[#6]:[#6]:*:[#6](-*):[#6]:2:1'}, 984: {'*=[#6](-*)-[#6]1:[#6]:[#6]:[#6]:*:[#6]:1:*'}, 1039: {'*-[#6]1:[#6]:[#6]:[#6]:[#6](:*):*:1'}, 1060: {'*=[#6](-[#7]-